In [2]:
import sys
from pathlib import Path

# Notebook 的工作目录通常是当前文件所在目录，向上查找服务根目录。
service_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
        and (path / "app" / "core" / "config.py").is_file()
    ),
    None,
)
if service_root is None:
    raise RuntimeError("未找到 agent-service 根目录，请从该项目或 notebooks 子目录运行。")

if str(service_root) not in sys.path:
    sys.path.insert(0, str(service_root))


In [3]:
# 1.查询产品及对应的条款
from app.infra.database import AsyncSessionFactory
from app.modules.product.service import ProductService

async with AsyncSessionFactory() as session:
    service = ProductService(session)
    products = await service.list_products(category=None)

for product in products:
    print(product) 

product = products[2]

Product({'insurer': '信美人寿相互保险社', 'name': '达尔文宝贝计划12号', 'clause_name': '信美相互星辰守护B款少儿重疾险条款.pdf', 'description': '117种重疾+28种中症+45种轻症，少儿重疾保障灵活。', 'max_premium': Decimal('2501.83'), 'highlights': ['117种重疾', '少儿保障', '可选投保人豁免'], 'created_at': '2026-11-12 12:16:08', 'category': 'critical_illness', 'id': 1, 'image_url': 'https://files.huizecdn.com/file3/M00/91/12/rBYA3Gi1gZ2AXBL7AAYFJxRhbPg411.png', 'min_premium': Decimal('285.30'), 'target_group': '0-17周岁', 'status': 'active', 'updated_at': '2026-11-12 12:16:08'})
Product({'insurer': '复星联合健康保险', 'name': '达尔文12号重大疾病保险', 'clause_name': '复星联合优选K重疾险(达尔文12号)条款.pdf', 'description': '120种重疾+30种中症+45种轻症，可选多项责任。', 'max_premium': Decimal('4815.00'), 'highlights': ['120种重疾', '中轻症保障', '可选责任丰富'], 'created_at': '2026-11-12 12:16:08', 'category': 'critical_illness', 'id': 2, 'image_url': 'https://files.huizecdn.com/file3/M00/90/E5/rBYA3Gi1BnWALIBtAAYFAsRjzwk030.png', 'min_premium': Decimal('561.00'), 'target_group': '28天-35周岁（1-4类职业）', 'status': 'active', 'u

In [5]:
# 2.解析文档
from app.core.config import settings
from mineru import MinerU

# 创建客户端
client = MinerU(token=settings.rag.mineru_token)
# 解析
result = client.extract(f"../../data/raw/kb/{product.clause_name}")

In [6]:
# 保存解析结果
result.save_markdown('output/r1.md')

PosixPath('output/r1.md')

In [8]:
# 3.文档优化，确保文档标题正确解析
from langchain.chat_models import init_chat_model

MARKDOWN_OPTIMIZATION_PROMPT = """
下面这份Markdown文档是从保险条款PDF解析得来，由于PDF中的各个小节是以表格形式存在，所以解析时出现错乱。
你分析内容，帮我转为格式正确的Markdown，特别是标题编号要正确。
- 标题等级要从1级标题开始，逐层增加，目录和文档名不计入标题等级。
- 输出结果中不要包含正文开始之前的部分。
- 输出结果只包含Markdown正文，不要解释或代码围栏。
- 不要修改保险条款。
""".strip()

llm = init_chat_model(
    "deepseek-flash",
    api_key=settings.llm.api_key,
    extra_body={'thinking': {'type': 'disabled'}},
    max_tokens=10000
)

response = llm.invoke([
    {"role": "system", "content": MARKDOWN_OPTIMIZATION_PROMPT},
    {"role": "user", "content": result.markdown},
])

if isinstance(response.content, str):
    markdown = response.content

In [8]:
# 保存文档
output_path = Path("output/r2.md")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(markdown, encoding="utf-8")

17540

In [9]:
from app.rag.models import ParentChunk
from langchain_classic.schema import Document
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

# 4.文档切分
# 4.1.markdown切分器，处理父块
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
        ("####", "Header 4"),
        ("#####", "Header 5"),
        ("######", "Header 6"),
    ],
    strip_headers=True,
)

# 4.2.递归切分器，处理子块
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", "。", "；", ";", "，", ","],
    keep_separator="end",
)

# 4.3.先用 markdown切分，再用递归切分
children_texts = markdown_splitter.split_text(markdown)
children_texts[0]

Document(metadata={'Header 1': '复星联合医联有盟重大疾病保险（互联网）条款'}, page_content='（本公司在每页底部对一些专业名词做了释义，这些释义为本条款的重要组成部分。同一专业名词在条款中出现多次的，本公司仅在该专业名词第一次出现的地方做了释义，该释义适用于全文。）')

In [10]:
# 4.4.处理子块
import uuid

parent_chunks: list[ParentChunk] = []
child_chunks: list[Document] = []

for section in children_texts:
    # 生成章节路径，section.metadata 就是每一级标题的字典
    session_path = list(section.metadata.values())
    # 组织 parent_chunk
    parent_chunk = ParentChunk(
        id = uuid.uuid4(),
        product_id = product.id,
        content=section.page_content,
        section_path=session_path,
        clause_name=product.clause_name
    )
    parent_chunks.append(parent_chunk)

    # 组织 section 的 metadata
    section.metadata = {
        "product_id": product.id,
        "parent_id": str(parent_chunk.id),
    }

    # 切分 section 得到子块
    children = text_splitter.split_documents([section])
    # 把父块的 section_path添加到子块的 metadata 中
    section_path_header = "\n".join(session_path) + "\n"
    for child in children:
        child.page_content = section_path_header + child.page_content
    child_chunks.extend(children)

In [11]:
# 5.子块进入向量数据库
from langchain_milvus import BM25BuiltInFunction, Milvus
from langchain_ollama import OllamaEmbeddings

# 创建向量模型，可以把文档转稠密向量
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024,  # 可选：减少维度以节省存储
    base_url="http://10.40.0.31:11434",  # 可选：指定 Ollama 服务器地址
)

# 初始化 vector store
vector_store = Milvus(
    embedding_function=embeddings,
    collection_name="insurance_collection",
    builtin_function=BM25BuiltInFunction(analyzer_params={"type": "chinese"}),
    vector_field=["dense", "sparse"],
    connection_args={"uri": settings.rag.milvus_url},
    auto_id=True,
)

# 删除旧文档
vector_store.delete(expr=f"product_id=={product.id}")

# 添加新文档
vector_store.add_documents(child_chunks)

[470413685495125140,
 470413685495125141,
 470413685495125142,
 470413685495125143,
 470413685495125144,
 470413685495125145,
 470413685495125146,
 470413685495125147,
 470413685495125148,
 470413685495125149,
 470413685495125150,
 470413685495125151,
 470413685495125152,
 470413685495125153,
 470413685495125154,
 470413685495125155,
 470413685495125156,
 470413685495125157,
 470413685495125158,
 470413685495125159,
 470413685495125160,
 470413685495125161,
 470413685495125162,
 470413685495125163,
 470413685495125164,
 470413685495125165,
 470413685495125166,
 470413685495125167,
 470413685495125168,
 470413685495125169,
 470413685495125170,
 470413685495125171,
 470413685495125172,
 470413685495125173,
 470413685495125174,
 470413685495125175,
 470413685495125176,
 470413685495125177,
 470413685495125178,
 470413685495125179,
 470413685495125180,
 470413685495125181,
 470413685495125182,
 470413685495125183,
 470413685495125184,
 470413685495125185,
 470413685495125186,
 470413685495

In [15]:
# 6.父知识块入库

from app.rag.repository import ParentChunkRepository

async with AsyncSessionFactory() as session, session.begin():
    repository = ParentChunkRepository(session)
    # 先删除对应的父块内容
    await repository.delete_by_product_id(product_id=product.id)
    # 批量全部添加
    repository.add_all(parent_chunks)

In [ ]:
# 7.测试根据id读取父类知识库
from app.infra.database import AsyncSessionFactory
from app.rag.repository import ParentChunkRepository

async with AsyncSessionFactory() as session, session.begin():
    repository = ParentChunkRepository(session)
    chunks = await repository.list_by_ids(["64ccb4c3-656e-419a-979f-30e8e4672dc6"])

if chunks is not None:
    for c in chunks:
        print(c)

ParentChunk({'clause_name': '信美相互星辰守护B款少儿重疾险条款.pdf', 'content': '在本条款中，“您”指投保人，“我们”指信美人寿相互保险社，“本合同”指您与我们之间订立的“信美相互互联网星辰守护B款少儿重大疾病保险合同”，投保人、被保险人的姓名在保险单上载明。', 'id': UUID('64ccb4c3-656e-419a-979f-30e8e4672dc6'), 'section_path': ['信美相互互联网星辰守护B款少儿重大疾病保险条款'], 'product_id': 1, 'created_at': '2026-11-15 12:30:08'})


In [16]:
# 8.检索
from app.rag.retrieval import ParentChunkRetrieval

retrieval = ParentChunkRetrieval(vector_store= vector_store)
parent_chunks = await retrieval.retrieve(
    query="保险合同的等待期是多久？",
    k=5,
    product_id=product.id,
)

for parent_chunk in parent_chunks:
    print("="*80)
    print("product_id =", parent_chunk.product_id)
    print("clause_name =", parent_chunk.clause_name)
    print("section_path =", parent_chunk.section_path)
    print("content =", parent_chunk.content)


2026-11-16 11:31:58 [info     ] RAG Retrieval 初始化完成✅          
product_id = 3
clause_name = 复星联合医联有盟重疾险条款.pdf
section_path = ['复星联合医联有盟重大疾病保险（互联网）条款', '2 提供的保障', '2.3 保险责任', '2.3.1 等待期']
content = 本合同生效日或本合同效力中止后的最后复效日（以较迟者为准）起90日为等待期。等待期是指本合同生效后本公司不承担保险责任的一段时间。  
（1）等待期内被保险人因非意外的原因确诊初次发生本合同所列的重大疾病<sup>6</sup>（一种或多种，下同），本公司不承担保险责任，但无息退还本合同已交保险费<sup>7</sup>，本公司已给付过一般医疗保险金的，还应扣除累计给付的相应金额，本合同终止；  
（2）等待期内被保险人罹患疾病或进行健康检查，因此所产生的医疗费用，无论此等费用是否发生在等待期内，本公司都不承担保险责任；  
（3）等待期内被保险人因非意外的原因确诊初次发生本合同所列的轻度疾病<sup>8</sup>（一种或多种，下同），中度疾病<sup>9</sup>（一种或多种，下同）本公司不承担且不再承担给付该种轻度疾病或中度疾病的保险金责任，本合同继续有效；  
（4）等待期内被保险人因非意外的原因导致身故或确定全残<sup>10</sup>的，本公司不承担保险责任，但无息退还本合同已交保险费，本公司已给付过一般医疗保险金的，还应扣除累计给付的相应金额，本合同终止。  
被保险人在保险期间内，因意外的原因导致发生本合同约定的保险事故的，无等待期。
product_id = 3
clause_name = 复星联合医联有盟重疾险条款.pdf
section_path = ['复星联合医联有盟重大疾病保险（互联网）条款', '1 合同订立', '1.4 犹豫期']
content = 自投保人签收本合同或收到本合同电子保单之日起（二者较早之日），投保人有15日的犹豫期。在此期间，请投保人认真审阅本合同，如果投保人认为本合同与投保人的需求不相符并在犹豫期内未发生理赔，可以在此期间提出解除本合同，本公司将向投保人无息退还保险费，本公司支付过保险赔款的，还应扣除相应赔款金额。犹豫期